# 01 - Data Cleaning (Python / Pandas)

**Goal:** turn the raw Superstore file into one clean `orders` table that I can analyse in SQL, Excel and Power BI.

Every step below says *what* I do and *why*. Nothing is dropped silently.

In [1]:
import sqlite3
import pandas as pd

raw = pd.read_csv("../data/raw/superstore_raw.csv")
print(raw.shape)
raw.drop(columns="Customer Name").head(3)

(10800, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.0,0.0,41.9136
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Consumer,United States,Henderson,Kentucky,42420.0,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3.0,0.0,219.5820
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Corporate,United States,Los Angeles,California,90036.0,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2.0,0.0,6.8714


## 1. First look: three sheets are stacked in one file

I expected about 10,000 rows, but there are 10,800. The original Superstore workbook has three sheets
(Orders, People, Returns) and this CSV has them stacked on top of each other. Rows from the other two
sheets have almost every column empty, so the first thing to do is split them.

In [2]:
# Real order rows have an Order Date. Everything else belongs to the other two sheets.
orders = raw[raw["Order Date"].notna()].copy()
leftover = raw[raw["Order Date"].isna()]

print("order rows:", len(orders), "| leftover rows:", len(leftover))

# rows 0-4 of the leftover are the People sheet (names), so I only print the header rows of each sheet
leftover[["Row ID", "Order ID"]].iloc[[0, 5, 6, 7]]

order rows: 9994 | leftover rows: 806


,Row ID,Order ID
9994,Person,Region
9999,Returned,Order ID
10000,Yes,CA-2015-100762
10001,Yes,CA-2015-100762


The leftover rows are the **People** sheet (regional manager names, not needed, so I ignore it) and the
**Returns** sheet (a list of returned order IDs, which is useful). The Returns rows start after the row
whose first cell says `Returned`.

In [3]:
start = leftover.index[leftover["Row ID"] == "Returned"][0]
returns = leftover.loc[start + 1:, ["Order ID"]]

print("return rows:", len(returns), "| unique returned orders:", returns["Order ID"].nunique())

# the same order is listed several times, so keep each order once
returns = returns.drop_duplicates()
print("all returned orders exist in the orders table:", returns["Order ID"].isin(orders["Order ID"]).all())

return rows: 800 | unique returned orders: 296
all returned orders exist in the orders table: True


## 2. Fix data types

- Dates are text, so I convert them with an explicit format (`month/day/year`). An explicit format avoids
  Pandas guessing day and month the wrong way round.
- `Postal Code` is an identifier, not a number. It was read as a decimal and lost its leading zero
  (05401 became 5401), so I turn it back into 5-character text.

In [4]:
orders["Row ID"] = orders["Row ID"].astype(int)
orders["Order Date"] = pd.to_datetime(orders["Order Date"], format="%m/%d/%Y")
orders["Ship Date"] = pd.to_datetime(orders["Ship Date"], format="%m/%d/%Y")
orders["Quantity"] = orders["Quantity"].astype(int)
orders["Postal Code"] = orders["Postal Code"].astype("Int64").astype("string").str.zfill(5)

orders.dtypes

Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code              string
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object

## 3. Missing values

In [5]:
missing = orders.isna().sum()
print(missing[missing > 0])

orders[orders["Postal Code"].isna()][["City", "State"]].drop_duplicates()

Postal Code    11
dtype: int64


,City,State
2234,Burlington,Vermont


**Decision:** only `Postal Code` has gaps (11 rows) and all of them are Burlington, Vermont. I do not use
postal codes in the analysis, and filling them in would mean guessing, so I leave them empty.

## 4. Duplicates

I check three levels of duplication, from strictest to loosest.

In [6]:
print("completely identical rows:", orders.duplicated().sum())
print("repeated Row ID:", orders["Row ID"].duplicated().sum())
print("same Order ID + Product ID:", orders.duplicated(["Order ID", "Product ID"]).sum())

same_line = orders[orders.duplicated(["Order ID", "Product ID"], keep=False)]
same_line.sort_values(["Order ID", "Product ID"])[["Row ID", "Order ID", "Product ID", "Sales", "Quantity", "Profit"]].head(6)

completely identical rows: 0
repeated Row ID: 0
same Order ID + Product ID: 8


,Row ID,Order ID,Product ID,Sales,Quantity,Profit
6498,6499,CA-2016-103135,OFF-BI-10000069,135.09,9,62.1414
6500,6501,CA-2016-103135,OFF-BI-10000069,90.06,6,41.4276
350,351,CA-2017-129714,OFF-PA-10001970,24.56,2,11.5432
352,353,CA-2017-129714,OFF-PA-10001970,49.12,4,23.0864
1300,1301,CA-2017-137043,FUR-FU-10003664,572.76,6,166.1004
1301,1302,CA-2017-137043,FUR-FU-10003664,286.38,3,83.0502


**Decision:** no row is duplicated and every `Row ID` is unique. Eight rows repeat an Order ID + Product ID.
Seven of those pairs have different quantities, so they are genuine split order lines. One pair
(rows 3406 and 3407) is identical apart from the Row ID, which could be a data-entry duplicate.
It is worth about 281 of 2.3 million in sales, and I cannot prove it is wrong, so I keep it and mention it
in the README instead of deleting data I cannot justify deleting.

## 5. Text clean-up and validation checks

I strip stray spaces from text columns, then check that the numbers make business sense.

In [7]:
text_columns = ["Order ID", "Ship Mode", "Customer ID", "Segment", "City", "State", "Region",
                "Product ID", "Category", "Sub-Category", "Product Name"]
for col in text_columns:
    orders[col] = orders[col].str.strip()

checks = {
    "Sales is always above 0": (orders["Sales"] > 0).all(),
    "Quantity is always above 0": (orders["Quantity"] > 0).all(),
    "Discount is between 0 and 1": orders["Discount"].between(0, 1).all(),
    "Ship Date is never before Order Date": (orders["Ship Date"] >= orders["Order Date"]).all(),
}
checks

{'Sales is always above 0': np.True_,
 'Quantity is always above 0': np.True_,
 'Discount is between 0 and 1': np.True_,
 'Ship Date is never before Order Date': np.True_}

In [8]:
print("rows with negative profit:", (orders["Profit"] < 0).sum())
print("Country values:", orders["Country"].unique())
print("Product IDs that have more than one name:", (orders.groupby("Product ID")["Product Name"].nunique() > 1).sum())

rows with negative profit: 1871
Country values: <StringArray>
['United States']
Length: 1, dtype: str
Product IDs that have more than one name: 32


Notes I will have to defend in an interview:

- **Negative profit is not an error.** 1,871 rows lose money. These are real loss-making sales, and they
  are exactly what the analysis is looking for, so I keep them.
- **Product ID is not a reliable product key.** 32 Product IDs have more than one name. I therefore rank
  products by `Product Name`, which is what a manager would recognise.
- `Country` is always "United States", so it carries no information.

## 6. New columns and privacy

- `Year`, `Order Month`, `Ship Days`, `Profit Margin` make later analysis simpler.
- `Returned` flags every line of an order that appears in the Returns sheet.
- I drop `Customer Name` (personal data that I do not need, `Customer ID` is enough) and `Country` (constant).

In [9]:
orders["Year"] = orders["Order Date"].dt.year
orders["Order Month"] = orders["Order Date"].dt.to_period("M").dt.to_timestamp()
orders["Ship Days"] = (orders["Ship Date"] - orders["Order Date"]).dt.days
orders["Profit Margin"] = orders["Profit"] / orders["Sales"]
orders["Returned"] = orders["Order ID"].isin(returns["Order ID"]).map({True: "Yes", False: "No"})

orders = orders.drop(columns=["Customer Name", "Country"])
orders.head(3)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Segment,City,State,Postal Code,...,Product Name,Sales,Quantity,Discount,Profit,Year,Order Month,Ship Days,Profit Margin,Returned
0,1,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Consumer,Henderson,Kentucky,42420,...,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136,2017,2017-11-01,3,0.16,No
1,2,CA-2017-152156,2017-11-08,2017-11-11,Second Class,CG-12520,Consumer,Henderson,Kentucky,42420,...,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820,2017,2017-11-01,3,0.30,No
2,3,CA-2017-138688,2017-06-12,2017-06-16,Second Class,DV-13045,Corporate,Los Angeles,California,90036,...,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714,2017,2017-06-01,4,0.47,No


## 7. Save the clean data

- A CSV for Excel and Power BI.
- A SQLite database for the SQL analysis. SQLite comes with Python, so there is nothing to install.
  I use snake_case column names there so queries do not need quotes around names like `Sub-Category`,
  and I store dates as `YYYY-MM-DD` text because SQLite has no date type.

In [10]:
orders.to_csv("../data/processed/orders_clean.csv", index=False, date_format="%Y-%m-%d")

orders_db = orders.copy()
for col in ["Order Date", "Ship Date", "Order Month"]:
    orders_db[col] = orders_db[col].dt.strftime("%Y-%m-%d")
orders_db.columns = orders_db.columns.str.lower().str.replace(" ", "_").str.replace("-", "_")

conn = sqlite3.connect("../data/superstore.db")
orders_db.to_sql("orders", conn, if_exists="replace", index=False)
print(pd.read_sql_query("SELECT COUNT(*) AS rows_in_database FROM orders", conn))
conn.close()

   rows_in_database
0              9994


## Summary of cleaning decisions

| Issue found | What I did |
|---|---|
| 3 sheets stacked in one CSV (806 non-order rows) | Split them; kept Returns as an order flag, ignored People |
| Dates stored as text | Converted with an explicit format |
| Postal codes lost leading zeros | Converted back to 5-character text |
| 11 missing postal codes | Left empty (not used, no safe way to fill) |
| 1 suspected duplicate order line | Kept and documented (about 281 in sales) |
| Customer names | Dropped for privacy |
| 1,871 negative-profit rows | Kept, they are real losses |